In [8]:
project_endpoint = "https://ice-foundry-01.services.ai.azure.com/api/projects/foundry-proj-01" # https://ice-foundry-01.services.ai.azure.com/api/projects/foundry-proj-01
eval_id = ""
eval_run_id = ""
overwrite = False

In [9]:
%pip install "azure-ai-projects>=2.2.0" azure-identity openai

In [10]:
import os
import json
import time
import datetime
from pprint import pprint
from pathlib import Path
from typing import Any, List, Dict, Iterable, Union
from packaging.version import Version

import pandas as pd

from azure.identity import DefaultAzureCredential, ClientSecretCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import DatasetVersion
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileID,
)

from collections.abc import Mapping
from azure.core.exceptions import ResourceNotFoundError

In [11]:
## SP for authentication to the Foundry Project
tenant_id = "94e2fbfd-171c-4f8c-9b70-fbe8002dac58"
client_id = "e6158f09-b7d6-426f-8613-711fd3d2c6f2"

connection_id = "415116a8-451f-4781-8329-cdaac05395b9" # connection name: "gelatai-in-keyvault admin"
client_secret = notebookutils.credentials.getSecretWithConnection(connection_id, "sp-fabriceval-secret")

In [12]:
def safe_getattr(obj, path, default=None):
    """
    Access nested attributes via dotted path, e.g. 'data_source.target.name'.
    Returns default if any segment is missing/None.
    Automatically unwraps single-element lists/tuples.
    """
    cur = obj
    for part in path.split('.'):
        if cur is None:
            return default

        while isinstance(cur, (list, tuple)) and len(cur) == 1:
            cur = cur[0]

        if isinstance(cur, dict):
            cur = cur.get(part, default)
        else:
            cur = getattr(cur, part, default)

    while isinstance(cur, (list, tuple)) and len(cur) == 1:
        cur = cur[0]

    return cur

def get_run_id(run):
    # tuple → unwrap
    if isinstance(run, tuple):
        run = run[0]

    # already a string → it's the id
    if isinstance(run, str):
        return run

    # object → extract id
    if hasattr(run, "id"):
        return run.id

    raise ValueError(f"Unsupported run type: {type(run)}")

In [13]:
from collections.abc import Iterable
import time

credential = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
client = project_client.get_openai_client()

def as_list(x):
    if x is None:
        return []
    # strings are iterable but should be treated as scalar values
    if isinstance(x, (str, bytes)):
        return [x]
    if isinstance(x, Iterable):
        try:
            return list(x)
        except TypeError:
            pass
    return [x]

def get_id(obj, field_name="id"):
    if hasattr(obj, field_name):
        return getattr(obj, field_name)
    if isinstance(obj, dict) and field_name in obj:
        return obj[field_name]
    return None

# 1. Get evals
if eval_id:
    eval_objects = [client.evals.retrieve(eval_id)]
else:
    eval_objects = list(client.evals.list())

all_output_items = []
all_runs = []
all_evals = []

for eval_object in eval_objects:
    current_eval_id = get_id(eval_object)
    if not current_eval_id:
        print(f"Skipping eval with unsupported shape: {type(eval_object)}")
        continue

    all_evals.append(eval_object)

    # 2. Get runs for this exact eval
    if eval_run_id:
        run_candidates = [client.evals.runs.retrieve(
            run_id=eval_run_id,
            eval_id=current_eval_id
        )]
    else:
        run_candidates = list(client.evals.runs.list(current_eval_id))

    for run_obj in as_list(run_candidates):
        run_id = get_id(run_obj)

        # Defensive handling only if your runtime really returns strings
        if run_id is None and isinstance(run_obj, str):
            run_id = run_obj

        if not run_id:
            print(f"Skipping run with unsupported shape: {type(run_obj)} -> {run_obj!r}")
            continue

        # 3. Re-retrieve using the SAME eval_id + run_id
        #    This validates the pair and gives you authoritative status.
        run = client.evals.runs.retrieve(
            run_id=run_id,
            eval_id=current_eval_id
        )

        # 4. Poll until terminal
        while run.status not in {"completed", "failed", "canceled"}:
            time.sleep(2)
            run = client.evals.runs.retrieve(
                run_id=run_id,
                eval_id=current_eval_id
            )

        all_runs.append(run) # should be run_items

        if run.status != "completed":
            print(f"Skipping output_items for run {run_id} because status={run.status}")
            continue

        # 5. Fetch output items only for completed runs
        output_items = list(
            client.evals.runs.output_items.list(
                run_id=run_id,
                eval_id=current_eval_id
            )
        )

        all_output_items.append({
            "eval_id": current_eval_id,
            "run_id": run_id,
            "items": output_items
        })

In [19]:
run_rows = []
criteria_rows = []

for run in all_runs:

    dt_utc = datetime.datetime.fromtimestamp(
        safe_getattr(run, "created_at"),
        tz=datetime.timezone.utc
    )

    agent_name = safe_getattr(run, "data_source.target.name")
    agent_version = safe_getattr(run, "data_source.target.version")
    
    agent = project_client.agents.get_version(agent_name=agent_name, agent_version=agent_version)
    agentModelName = agent["definition"]["model"]
    deployment = project_client.deployments.get(name=agentModelName)
    modelName = deployment["modelName"]
    modelVersion = deployment["modelVersion"]

    run_rows.append({
        "id": safe_getattr(run, "id"),
        "eval_id": safe_getattr(run, "eval_id"),
        "run_name": safe_getattr(run, "name"),
        "created_at": dt_utc,
        "target_type": safe_getattr(run, "data_source.target.type"),
        "target_name": agent_name,
        "target_version": agent_version,
        "target_model_name": agentModelName,
        "target_model": modelName,
        "target_model_version": modelVersion,
        "data_source_type": safe_getattr(run, "data_source.type"),
        "report_url": safe_getattr(run, "report_url"),
        "status": safe_getattr(run, "status"),
        #"created_at": safe_getattr(run, "created_at"),
    })
    
    for crit in run.per_testing_criteria_results:
            criteria_rows.append({
                "evalrun_id": run.id,
                "testing_criteria": crit.testing_criteria,
                "failed": crit.failed,
                "passed": crit.passed
            })


df_runs = pd.DataFrame(run_rows)
df_runs["created_at"] = df_runs["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
df_criteria = pd.DataFrame(criteria_rows)
# display(df_runs)
# display(df_criteria)

In [21]:
run_itemresult_rows = []
run_outputitem_rows = []

for run in all_output_items:
    items = run.get("items", [])
    for evaluation in items:
        run_outputitem_rows.append({
            "id": safe_getattr(evaluation, "id"),
            "output_item_id": safe_getattr(evaluation, "id") + "-" + safe_getattr(evaluation, "run_id"),
            "eval_id": safe_getattr(evaluation, "eval_id"),
            "run_id": safe_getattr(evaluation, "run_id"),
            # "cached_tokens": safe_getattr(evaluation, "usage.cached_tokens"),
            # "completion_tokens": safe_getattr(evaluation, "usage.completion_tokens"),
            # "prompt_tokens": safe_getattr(evaluation, "usage.prompt_tokens"),
            # "total_tokens": safe_getattr(evaluation, "usage.total_tokens"),
            "status": safe_getattr(evaluation, "status"),
            # "query": safe_getattr(evaluation, "datasource_item.query"),
            # "context": safe_getattr(evaluation, "datasource_item.context"),
            # "ground_truth": safe_getattr(evaluation, "datasource_item.ground_truth"),
            # "response": safe_getattr(evaluation, "datasource_item.sample.output_text"),
        })

        for result in evaluation.results:
            run_itemresult_rows.append({
                "output_item_id": safe_getattr(evaluation, "id") + "-" + safe_getattr(evaluation, "run_id"),
                "name": safe_getattr(result, "name"),
                "passed": safe_getattr(result, "passed"),
                "score": safe_getattr(result, "score"),
                "reason": safe_getattr(result, "reason"),
                "threshold": safe_getattr(result, "threshold"),
            })

df_run_output_result_item = pd.DataFrame(run_itemresult_rows)
df_run_output_result_item["score"] = pd.to_numeric(df_run_output_result_item["score"], errors="coerce").fillna(0.0)
df_run_output_result_item["passed"] = df_run_output_result_item["passed"].astype("string").fillna("NA")
df_run_output_result_item["reason"] = df_run_output_result_item["reason"].fillna("")

df_run_output_item = pd.DataFrame(run_outputitem_rows)

#display(df_run_result_item)
#display(df_run_output_item)

In [22]:
import pandas as pd
from deltalake import write_deltalake

if overwrite:
    writeMode = "overwrite"
else:
    writeMode = "append"

print(f"Using write mode: {writeMode}")
delta_table_path = "/lakehouse/default/Tables/dbo/runs"
write_deltalake(delta_table_path, df_runs, mode=writeMode, schema_mode='merge', engine='rust', storage_options={"allow_unsafe_rename": "true"})

delta_table_path = "/lakehouse/default/Tables/dbo/run_criteria"
write_deltalake(delta_table_path, df_criteria, mode=writeMode, schema_mode='merge', engine='rust', storage_options={"allow_unsafe_rename": "true"})

delta_table_path = "/lakehouse/default/Tables/dbo/run_output_item"
write_deltalake(delta_table_path, df_run_output_item, mode=writeMode, schema_mode='merge', engine='rust', storage_options={"allow_unsafe_rename": "true"})

delta_table_path = "/lakehouse/default/Tables/dbo/run_output_result_item"
write_deltalake(delta_table_path, df_run_output_result_item, mode=writeMode, schema_mode='merge', engine='rust', storage_options={"allow_unsafe_rename": "true"})